# Final Notebook - House Price Prediction

Notebook này tổng hợp toàn bộ chương trình dự đoán giá nhà: lấy dữ liệu Kaggle, kiểm tra chất lượng dữ liệu, EDA, preprocessing, feature engineering, training, tuning, đánh giá tối ưu, inference demo và giao diện HTML.

In [ ]:
# Setup dependencies and imports from requirements.txt
# Run this cell first in Jupyter Notebook or Google Colab.
import importlib.util
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

requirements_path = PROJECT_ROOT / 'requirements.txt'
package_imports = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'scikit-learn': 'sklearn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'joblib': 'joblib',
    'streamlit': 'streamlit',
    'flask': 'flask',
    'kagglehub': 'kagglehub',
    'ipython': 'IPython',
    'pytest': 'pytest',
}
missing = [pkg for pkg, module in package_imports.items() if importlib.util.find_spec(module) is None]
if missing and requirements_path.exists():
    print('Installing missing packages from requirements.txt:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)])
elif missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('All required packages are already installed.')

import json
import warnings

import joblib
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from flask import Flask
from IPython.display import Image, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore', category=FutureWarning)
print('Imports ready. Project root:', str(PROJECT_ROOT).encode('ascii', errors='replace').decode('ascii'))


## 0. Nội dung được tổng hợp

- Bài toán regression dự đoán `SalePrice`.
- Nguồn dữ liệu thô chỉ từ Kaggle và cách chạy trên Google Colab.
- Hình ảnh EDA/model giống cấu trúc báo cáo spam email: distribution, scatter, correlation, model comparison, residuals, learning curve.
- Model selection dựa trên RMSE, MAE, R2 và kết quả tuning.

## 1. Problem Definition

Mục tiêu là xây dựng mô hình Machine Learning dự đoán giá bán nhà. Mỗi dòng dữ liệu tương ứng một căn nhà; các cột đầu vào mô tả diện tích, chất lượng, năm xây dựng, garage, khu vực và điều kiện bán; target là `SalePrice`.

Đây là bài toán **supervised learning - regression** vì output là giá trị số liên tục.

## 1.1. Đối chiếu yêu cầu bài 5.2 - House Price Prediction

**Mô tả:** Sử dụng Linear Regression để dự đoán giá của các căn nhà dựa trên các yếu tố như diện tích, số phòng, vị trí, chất lượng nhà, điều kiện bán và các thông tin liên quan.

**Dữ liệu:** Dữ liệu thô được thu thập từ các nguồn Kaggle về bất động sản/giá nhà. Sau khi thu thập, dữ liệu được chuẩn hóa về schema chung gồm các nhóm thông tin chính: diện tích (`LivingArea`, `LotArea`), số phòng (`Bedrooms`, `Bathrooms`), vị trí (`Neighborhood`, `DatasetSource`), chất lượng/điều kiện (`QualityScore`, `ConditionScore`) và giá bán thực tế (`SalePrice`).

**Ghi chú:** Python dùng thư viện `scikit-learn`; lớp `LinearRegression` được dùng làm baseline đúng yêu cầu đề. Ngoài baseline này, notebook vẫn giữ thêm Ridge/Lasso/RandomForest/ExtraTrees/GradientBoosting/HistGradientBoosting để so sánh và chọn model tốt nhất.


## 1.2. Checklist yêu cầu trong ảnh

1. **Tiền xử lý dữ liệu:** Đọc và khám phá dữ liệu, lọc missing/duplicate/invalid labels/outliers, chuẩn hóa dữ liệu và chọn đặc trưng phù hợp.
2. **Chia tập dữ liệu:** Chia dữ liệu thành train/test bằng `train_test_split`.
3. **Xây dựng Linear Regression:** Huấn luyện baseline `LinearRegression` bằng dữ liệu train.
4. **Đánh giá mô hình:** Đánh giá trên test set bằng RMSE, MAE và R2.
5. **Tinh chỉnh mô hình:** Linear Regression thường không có `alpha`; notebook tinh chỉnh `alpha` qua Ridge/Lasso và tune thêm các model boosting để cải thiện hiệu suất.
6. **Dự đoán:** Dùng pipeline/model tốt nhất để dự đoán giá nhà mới qua notebook và HTML UI.


## 2. Success Metrics

- **RMSE**: lỗi lớn bị phạt mạnh, phù hợp khi sai số giá nhà lớn là nghiêm trọng.
- **MAE**: dễ giải thích vì là sai số trung bình theo đơn vị tiền.
- **R2**: cho biết model giải thích được bao nhiêu phương sai của `SalePrice`.
- Model cuối được chọn theo RMSE thấp nhất, sau đó kiểm tra MAE/R2 để tránh chọn model mất cân bằng.

## 3. Data Collection - Kaggle và Google Colab

Nguồn Kaggle được thu thập:

- https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset
- https://www.kaggle.com/datasets/yasserh/housing-prices-dataset
- https://www.kaggle.com/datasets/amitabhajoy/bengaluru-house-price-data
- https://www.kaggle.com/datasets/harlfoxem/housesalesprediction

Notebook dùng `kagglehub` để tải các Kaggle dataset public. Nếu muốn dùng competition `house-prices-advanced-regression-techniques`, upload `kaggle.json` trên Colab rồi đặt `train.csv` vào `data/raw/train.csv`.

In [ ]:
# Colab setup tham khảo. Chạy cell này khi mở notebook trên Google Colab.
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle data/raw
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle kagglehub pandas numpy scikit-learn matplotlib seaborn joblib flask
# !kaggle competitions download -c house-prices-advanced-regression-techniques -p data/raw
# !unzip -o data/raw/house-prices-advanced-regression-techniques.zip -d data/raw

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('c:/Users/User/OneDrive - ut.edu.vn/Documents/Machine Learning - Trường/House-Price-Prediction')

## 4. Data Loading

Loader ưu tiên `data/processed/kaggle_house_prices_clean_labeled.csv`, được tạo từ nhiều nguồn Kaggle thô. Workflow hiện tại không dùng OpenML và không dùng sample data.

In [6]:
import pandas as pd
from IPython.display import Image, display

from config import EDGE_CASE_REPORT, FIGURES_DIR, METRICS_PATH, MODEL_PATH, TARGET_COLUMN
from src.data_loader import load_dataset, resolve_dataset_path, split_features_target
from src.data_quality import summarize_quality
from src.data_pipeline import prepare_real_dataset
from src.feature_engineering import add_house_features
from src.predict import predict_price
from src.train import train_all

prepared_df = prepare_real_dataset()
prepared_df['DatasetSource'].value_counts()

dataset_path = resolve_dataset_path()
df = load_dataset(dataset_path)
print('Dataset path:', dataset_path)
print('Shape:', df.shape)
df.head()

ModuleNotFoundError: No module named 'kagglehub'

## 4.1. Các trường dữ liệu chính theo yêu cầu

| Nhóm thông tin | Cột sử dụng trong notebook | Ý nghĩa |
| --- | --- | --- |
| Diện tích | `LivingArea`, `LotArea` | Quy mô căn nhà/khu đất |
| Số phòng | `Bedrooms`, `Bathrooms` | Nhu cầu sử dụng và tiện nghi cơ bản |
| Vị trí | `Neighborhood`, `DatasetSource` | Khu vực/thị trường dữ liệu |
| Chất lượng | `QualityScore`, `ConditionScore` | Chất lượng xây dựng/tình trạng căn nhà |
| Giá bán thực tế | `SalePrice` | Label/target cần dự đoán |

`DatasetSource` và `PriceUnit` được giữ lại để tránh hiểu sai khi gặp nhiều nguồn Kaggle có đơn vị tiền khác nhau.


## 5. EDA - Phân tích dữ liệu

EDA tập trung vào `SalePrice`, quan hệ giữa diện tích và giá, phân bố theo nguồn Kaggle, cùng các feature numeric có tương quan mạnh với giá nhà.

In [ ]:
df[TARGET_COLUMN].describe()

### Hình 1 - Kaggle rows by source

![Kaggle rows by source](../reports/figures/kaggle_rows_by_source.png)

### Hình 2 - SalePrice distribution

![SalePrice distribution](../reports/figures/saleprice_distribution.png)

### Hình 3 - SalePrice by Kaggle source

![SalePrice by Kaggle source](../reports/figures/saleprice_by_kaggle_source.png)

### Hình 4 - Scatter feature vs SalePrice

![Living area vs SalePrice by source](../reports/figures/living_area_vs_saleprice_by_source.png)

### Hình 5 - Correlation heatmap

![Correlation heatmap](../reports/figures/saleprice_correlation_heatmap.png)

## 6. Data Quality, Label Filtering và Edge Cases

Kiểm tra missing values, duplicate rows, numeric/categorical columns, target missing, label không hợp lệ và các trường hợp đặc biệt khi inference.

In [ ]:
quality_report = summarize_quality(df)
quality_report

## 7. Preprocessing

Pipeline dùng `ColumnTransformer` để xử lý dữ liệu đúng cách:

- Numeric: median imputation + standard scaling.
- Categorical: most frequent imputation + one-hot encoding.
- `handle_unknown='ignore'` giúp inference không lỗi khi gặp category mới.

## 8. Feature Engineering

Các feature bổ sung:

- `HouseAgeAtRemodel = YearRemodAdd - YearBuilt`.
- `AreaPerBedroom = GrLivArea / BedroomAbvGr`.
- `GarageAreaPerCar = GarageArea / GarageCars`.

Các feature này chỉ được tạo nếu cột nguồn tồn tại, nên notebook vẫn chạy được với dataset thay thế.

In [ ]:
df_features = add_house_features(df)
X, y = split_features_target(df_features)
print('Feature shape:', X.shape)
print('Target shape:', y.shape)
X.head()

## 9. Model Training

Các model được train:

- Linear Regression baseline.
- Ridge Regression.
- Lasso Regression.
- Random Forest Regressor.
- Gradient Boosting Regressor.
- Extra Trees Regressor.
- Hist Gradient Boosting Regressor.

Sau đó notebook chạy tuning nhẹ bằng cross-validation cho Gradient Boosting và Hist Gradient Boosting. Tuning dùng sample đại diện rồi fit lại best estimator trên toàn bộ train set.

In [ ]:
result = train_all(dataset_path=dataset_path, save_artifacts=True)
metrics = result['metrics']
tuning_report = result['tuning_report']
print('Best model:', result['best_model_name'])
print('Optimization status:', result['optimization_status'])
metrics

## 9.1. Baseline Linear Regression đúng yêu cầu đề

Cell này minh họa riêng mô hình `LinearRegression` theo đúng yêu cầu trong ảnh: chia train/test, huấn luyện trên train set và đánh giá trên test set. Kết quả baseline này được dùng để so sánh với các model tinh chỉnh ở phần sau.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from config import RANDOM_STATE, TEST_SIZE
from src.evaluate import regression_metrics
from src.preprocessing import build_preprocessor, infer_feature_types

numeric_features, categorical_features = infer_feature_types(X)
linear_preprocessor = build_preprocessor(numeric_features, categorical_features)
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
linear_regression_pipeline = Pipeline([
    ('preprocessor', linear_preprocessor),
    ('model', LinearRegression()),
])
linear_regression_pipeline.fit(X_train_lr, y_train_lr)
linear_predictions = linear_regression_pipeline.predict(X_test_lr)
linear_regression_metrics = regression_metrics(y_test_lr, linear_predictions)
linear_regression_metrics


### Ghi chú về tham số `alpha`

`LinearRegression` thuần không có tham số `alpha`. Khi đề yêu cầu thử hệ số điều chỉnh `alpha`, cách triển khai phù hợp trong `scikit-learn` là dùng Ridge/Lasso: Ridge dùng L2 regularization, Lasso dùng L1 regularization. Phần `Hyperparameter Tuning` bên dưới thực hiện đúng ý này.


## 10. Model Metrics - So sánh kết quả

Bảng metric được lưu tại `reports/metrics/model_metrics.csv`. Hình dưới giúp so sánh RMSE giữa các model.

![Model RMSE comparison](../reports/figures/model_rmse_comparison.png)

In [ ]:
pd.read_csv(METRICS_PATH)

### Nhận xét so sánh model

- Model có RMSE thấp nhất được chọn làm model cuối.
- Kết quả hiện tại được train trên Kaggle-only filtered data, không dùng OpenML/sample dataset.
- Nếu tuned model không vượt baseline trên test set, kết luận là baseline hiện đang phù hợp hơn với dữ liệu đang dùng.

## 11. Hyperparameter Tuning và kiểm tra tối ưu

Tuning dùng cross-validation với scoring `neg_root_mean_squared_error`. Báo cáo này trả lời yêu cầu kiểm tra model đã tối ưu chưa.

In [ ]:
tuning_report

## 12. Actual vs Predicted

Hình này tương tự phần đánh giá model của spam email: thay vì confusion matrix, bài toán regression dùng biểu đồ actual-vs-predicted.

![Actual vs predicted](../reports/figures/actual_vs_predicted_saleprice.png)

## 13. Residual Analysis

Residual giúp kiểm tra model còn sai lệch hệ thống hay không. Nếu residual phân bố quanh 0 và không tạo pattern rõ, model ổn hơn.

![Residual distribution](../reports/figures/residual_distribution.png)

![Residuals vs predicted](../reports/figures/residuals_vs_predicted.png)

## 14. Learning Curve

Learning curve cho biết khi tăng số lượng dữ liệu train, train RMSE và validation RMSE thay đổi thế nào.

![Learning curve](../reports/figures/learning_curve_best_model.png)

### Nhận xét learning curve

- Train RMSE thấp nhưng validation RMSE cao: model có dấu hiệu overfitting.
- Cả train và validation RMSE cao: model underfitting hoặc thiếu feature tốt.
- Hai đường hội tụ ở mức thấp: model ổn hơn.
- Kết luận cuối nên dựa trên Kaggle-only data đã lọc; nếu thay dataset, cần chạy lại toàn bộ pipeline để sinh lại metric và figures.

## 15. Model Selection

Model cuối được lưu vào `models/house_price_pipeline.joblib`. Artifact này chứa pipeline preprocessing, model tốt nhất, danh sách feature và metric để inference dùng đúng cấu trúc training.

In [ ]:
print('Saved model:', MODEL_PATH)
print('Best model:', result['best_model_name'])
print('Optimization status:', result['optimization_status'])

## 16. Demo Predict nhà mới

Demo lấy một dòng input, bỏ target, rồi predict `SalePrice` bằng pipeline đã lưu.

In [ ]:
demo_input = X.head(1)
prediction = predict_price(demo_input)
pd.DataFrame({'PredictedSalePrice': prediction})

## 17. HTML UI dự đoán giá nhà

Giao diện HTML nằm ở `templates/index.html`, backend Flask nằm ở `app_html.py`.

Chạy giao diện:

```powershell
python app_html.py
```

Sau đó mở `http://127.0.0.1:5000` để nhập thông tin nhà và nhận giá dự đoán.

## 17.1. So sánh Python UI và HTML UI

| Tiêu chí | Python UI bằng Streamlit | HTML UI bằng Flask |
| --- | --- | --- |
| Phù hợp demo Machine Learning | Rất phù hợp vì hiển thị dataframe, chart, metrics nhanh | Phù hợp nếu cần web form tối giản |
| Nguy cơ lỗi font tiếng Việt | Thấp hơn vì toàn bộ UI chạy trong Python/Streamlit | Cần kiểm soát `charset=utf-8` trong template và response header |
| Khả năng mở rộng báo cáo | Dễ thêm tabs Dataset, Metrics, Figures, Edge Cases | Cần viết thêm HTML/CSS/Jinja |
| Vai trò trong project này | Giao diện chính khuyến nghị | Fallback để giữ chức năng cũ |

Kết luận: với yêu cầu coursework/demo local, Streamlit tối ưu hơn HTML thủ công vì ít code giao diện hơn, giảm rủi ro font và bám sát workflow Python của notebook.


## 18. Code Standard và kiểm soát chất lượng

- Code được tách module theo `src/`: loader, quality, preprocessing, feature engineering, train, evaluate, predict.
- Test nằm trong `tests/`.
- Notebook parse bằng JSON chuẩn.
- Model, metrics và figures đều được sinh tự động khi chạy `python -m src.train`.

## 19. Kết luận cuối

Dự án đã có đủ workflow Machine Learning từ Kaggle-only data collection đến deployment giao diện HTML. Với dữ liệu Kaggle tổng hợp đã lọc, model tốt nhất hiện tại là `hist_gradient_boosting_tuned` theo RMSE/R2; notebook sẽ tự cập nhật kết luận khi chạy lại pipeline.

## 20. Checklist trình bày

- Có link Kaggle và hướng dẫn Colab.
- Có data collection và data quality report.
- Có lọc dữ liệu rác, lọc label và đánh nhãn dữ liệu.
- Có EDA figures.
- Có preprocessing và feature engineering.
- Có model comparison và hyperparameter tuning.
- Có residual analysis và learning curve.
- Có demo prediction.
- Có Python UI bằng Streamlit với tabs Predict/Dataset/Metrics/Figures/Edge Cases.
- Có HTML UI fallback dùng model đã train.
